# Parse an ASPECT log

`AspectLogParser` extracts timestep headers from an ASPECT log. It can stream records one at a time, which keeps memory use nearly constant for large logs, or collect all records into a list.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from gdmate.aspect_tools import AspectLogParser

## Create a small example log

We first use a deliberately small log to show the parser behavior without the unrelated solver output found in a full ASPECT log. It contains a valid ASPECT version header and three timestep headers. For a model run, replace `log_path` below with the path to the ASPECT standard-output log.

In [ ]:
temporary_directory = TemporaryDirectory()
log_path = Path(temporary_directory.name) / "log.txt"
log_path.write_text(
    """This is ASPECT
--     . version 3.1.0-pre (main, abc123)
*** Timestep 0:  t=0 years, dt=0 years
   Solving temperature system... 0 iterations.
*** Timestep 1:  t=100 years, dt=100 years
   Solving temperature system... 8 iterations.
*** Timestep 2:  t=2.5e+02 years, dt=1.5e+02 years
""",
    encoding="utf-8",
)

## Stream timestep records

`parser` is an `AspectLogParser`, not a list of timesteps. It is iterable, so the `for` loop asks it to read the file one line at a time and yield an `AspectTimestep` for every header such as `*** Timestep 1: t=100 years, dt=100 years`. Each returned record contains the timestep number, simulation time, timestep size, time unit, and source line number. Streaming is the preferred interface for large logs because previously returned records are not retained by the parser.

In [ ]:
parser = AspectLogParser(log_path)
for timestep in parser:
    print(
        f"step={timestep.number}, time={timestep.time:g} "
        f"{timestep.time_unit}, dt={timestep.timestep_size:g}, "
        f"line={timestep.line_number}"
    )

## Collect all records

Use `parse()` when the complete timestep history is convenient and small enough to retain in memory.

In [ ]:
timesteps = parser.parse()
timesteps

In [ ]:
times = [record.time for record in timesteps]
timestep_sizes = [record.timestep_size for record in timesteps]
times, timestep_sizes

Each immutable record also contains the time unit and source line number. A malformed timestep header raises a `ValueError` that identifies its file and line.

In [ ]:
temporary_directory.cleanup()

## Parse the provided test log

The small example above is useful for introducing the API, but the fixture below is more like real ASPECT output. It was reduced from an actual model log while retaining the ASPECT version header, two timestep headers, and two complete wall-clock timing tables. Change only `test_log_path` to analyze another log.

In [ ]:
test_log_path = Path(
    "../../tests/data/aspect_tools/aspect_log_with_timing_blocks.txt"
).resolve()
test_log_path

In [ ]:
test_parser = AspectLogParser(test_log_path)
print(
    f"ASPECT version {test_parser.version} "
    f"(major version {test_parser.major_version})"
)
test_timesteps = test_parser.parse()

print(f"Parsed {len(test_timesteps)} timestep records")
print(f"First: {test_timesteps[0]}")
print(f"Last:  {test_timesteps[-1]}")

## Parse wall-clock timing blocks

ASPECT prints a wall-clock summary beginning with a line such as `| Total wallclock time elapsed since start | 6.31e+03s |`. The parser returns one `AspectTimingBlock` for each complete summary table. A block contains the total wall-clock time, its source line, the most recent timestep number, and a tuple of `AspectTimingSection` records. Each section record represents one table row and contains its name, call count, wall time in seconds, and percentage of the total. A concatenated or restarted log can contain more than one timing block.

In [ ]:
timing_blocks = test_parser.parse_timing_blocks()
print(f"Parsed {len(timing_blocks)} timing blocks")

for block in timing_blocks:
    print(
        f"timestep={block.timestep_number}, "
        f"total={block.total_wallclock_seconds:g} s, "
        f"sections={len(block.sections)}, line={block.line_number}"
    )

Inspect the individual section rows in the most recent timing block:

In [ ]:
latest_timing_block = timing_blocks[-1]
for section in latest_timing_block.sections:
    print(
        f"{section.name}: calls={section.call_count}, "
        f"wall time={section.wall_time_seconds:g} s, "
        f"total={section.percent_of_total:g}%"
    )